In [ ]:
import os

GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY


In [ ]:
# Programmatically add file in bucket

# pip install google-cloud-storage

from google.cloud import storage
import os

def upload_file_to_gcs(local_file_path, bucket_name, destination_blob_path):
    """
    Upload a local file to Google Cloud Storage.

    Args:
        local_file_path: Path to the file on your local machine
        bucket_name: GCS bucket name
        destination_blob_path: Path in the bucket (e.g., 'input/myfile.pdf')
    """
    # Initialize the client (uses credentials from GOOGLE_APPLICATION_CREDENTIALS env var or ADC)
    storage_client = storage.Client()

    # Get the bucket
    bucket = storage_client.bucket(bucket_name)

    # Create a blob object (represents the file in GCS)
    blob = bucket.blob(destination_blob_path)

    # Upload the file
    blob.upload_from_filename(local_file_path)

    print(f"File {local_file_path} uploaded to gs://{bucket_name}/{destination_blob_path}")


# --- Example Usage ---
local_file = "Tafheemat1.pdf"
bucket = "my-tickrbot-bucket"
destination_path = "input/Tafheemat1.pdf"  # folder inside the bucket

upload_file_to_gcs(local_file, bucket, destination_path)



# Make sure you have authenticated with Google Cloud

In [ ]:
import os
import time
from google.cloud import documentai_v1 as documentai
from google.cloud import storage

def store_document(path: str, doc_content: str):
    """Save content to a local text file."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(doc_content)

def fetch_batch_output(storage_client, output_bucket, output_prefix):
    """Fetch JSON output files from GCS and combine text per page."""
    parsed_pages = []

    blobs = storage_client.list_blobs(output_bucket, prefix=output_prefix)
    blobs = sorted(blobs, key=lambda b: b.name)  # sort by filename

    for blob in blobs:
        if not blob.name.endswith(".json"):
            continue
        json_bytes = blob.download_as_bytes()
        doc = documentai.Document.from_json(json_bytes, ignore_unknown_fields=True)

        for page in doc.pages:
            page_number = page.page_number or 1
            page_text = ""
            if hasattr(page, "layout") and page.layout and hasattr(page.layout, "text_anchor"):
                segments = page.layout.text_anchor.text_segments
                for seg in segments:
                    start_idx = seg.start_index or 0
                    end_idx = seg.end_index or len(doc.text)
                    page_text += doc.text[start_idx:end_idx]
            if not page_text.strip():
                page_text = "[BLANK PAGE]"

            # Split each page clearly
            parsed_pages.append(f"\n--- PAGE {page_number} ---\n{page_text.strip()}\n--- END PAGE {page_number} ---\n")

    return "\n".join(parsed_pages)

def batch_process_document(
    project_id: str,
    location: str,
    processor_id: str,
    gcs_input_uri: str,
    gcs_output_uri: str,
    output_txt_path: str,
):
    """OCR a document stored in GCS using BatchProcessRequest, and save text per page."""
    # Set API endpoint
    client_options = {"api_endpoint": f"{location}-documentai.googleapis.com"}
    client = documentai.DocumentProcessorServiceClient(client_options=client_options)

    # Storage client using ADC + project
    storage_client = storage.Client(project=project_id)

    # Processor path
    processor_name = client.processor_path(project_id, location, processor_id)

    # Input config: single document in GCS
    gcs_doc = documentai.GcsDocument(gcs_uri=gcs_input_uri, mime_type="application/pdf")
    input_docs = documentai.GcsDocuments(documents=[gcs_doc])
    input_config = documentai.BatchDocumentsInputConfig(gcs_documents=input_docs)

    # Output config: GCS folder
    gcs_output_config = documentai.DocumentOutputConfig.GcsOutputConfig(gcs_uri=gcs_output_uri)
    output_config = documentai.DocumentOutputConfig(gcs_output_config=gcs_output_config)

    # Batch process request
    request = documentai.BatchProcessRequest(
        name=processor_name,
        input_documents=input_config,
        document_output_config=output_config,
    )

    print("Starting batch processing...")
    operation = client.batch_process_documents(request=request)
    operation.result(timeout=600)  # Wait until done
    print("Batch processing complete.")

    # Parse bucket/prefix from GCS output URI
    if not gcs_output_uri.startswith("gs://"):
        raise ValueError("GCS URI must start with gs://")
    parts = gcs_output_uri[5:].split("/", 1)
    bucket_name = parts[0]
    prefix = parts[1] if len(parts) > 1 else ""

    print("Fetching and combining output JSON files...")
    combined_text = fetch_batch_output(storage_client, bucket_name, prefix)

    # Save final OCR text
    store_document(output_txt_path, combined_text)
    print(f"OCR text saved to: {output_txt_path}")


# --- CONFIGURATION ---
PROJECT_ID = "your-gcp-project-id"
LOCATION = "us"
PROCESSOR_ID = "your-processor-id"
GCS_INPUT_URI = "gs://your-bucket/Tafheemat1.pdf"
GCS_OUTPUT_URI = "gs://your-bucket/output/"
OUTPUT_TXT_PATH = "output/Tafheemat1-ocr.txt"

# --- RUN ---
if __name__ == "__main__":
    # Ensure ADC is set with quota project:
    # export GOOGLE_APPLICATION_CREDENTIALS="~/.config/gcloud/application_default_credentials.json"
    batch_process_document(
        project_id=PROJECT_ID,
        location=LOCATION,
        processor_id=PROCESSOR_ID,
        gcs_input_uri=GCS_INPUT_URI,
        gcs_output_uri=GCS_OUTPUT_URI,
        output_txt_path=OUTPUT_TXT_PATH,
    )


In [ ]:
import io
from concurrent.futures import ThreadPoolExecutor
from google.cloud import documentai_v1 as documentai
from google.cloud import storage
from PyPDF2 import PdfReader, PdfWriter
import os

# --------------------------
# CONFIGURATION
# --------------------------
PROJECT_ID = "your-gcp-project-id"
LOCATION = "us"
PROCESSOR_ID = "your-processor-id"
GCS_INPUT_URI = "gs://your-bucket/Tafheemat1.pdf"
OUTPUT_TXT_PATH = "output/Tafheemat1-ocr.txt"

MAX_WORKERS = 6
PAGES_PER_CHUNK = 15

# --------------------------
# UTILITIES
# --------------------------
def store_document(path: str, doc_content: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(doc_content)

def parse_documentai_doc(doc):
    """Extract text from Document AI document per page."""
    parsed_pages = []
    for page in doc.pages:
        page_number = page.page_number or 1
        page_text = ""
        if hasattr(page, "layout") and page.layout and hasattr(page.layout, "text_anchor"):
            segments = page.layout.text_anchor.text_segments
            for seg in segments:
                start_idx = seg.start_index or 0
                end_idx = seg.end_index or len(doc.text)
                page_text += doc.text[start_idx:end_idx]
        if not page_text.strip():
            page_text = "[BLANK PAGE]"
        parsed_pages.append(f"\n--- PAGE {page_number} ---\n{page_text.strip()}\n--- END PAGE {page_number} ---\n")
    return "\n".join(parsed_pages)

def process_pdf_chunk_bytes(chunk_bytes):
    """Send PDF chunk bytes to Document AI using raw_document"""
    client_options = {"api_endpoint": f"{LOCATION}-documentai.googleapis.com"}
    client = documentai.DocumentProcessorServiceClient(client_options=client_options)
    processor_name = client.processor_path(PROJECT_ID, LOCATION, PROCESSOR_ID)

    raw_doc = documentai.RawDocument(content=chunk_bytes, mime_type="application/pdf")
    request = documentai.ProcessRequest(name=processor_name, raw_document=raw_doc)
    doc = client.process_document(request=request).document
    return parse_documentai_doc(doc)


# --------------------------
# SPLIT PDF IN MEMORY
# --------------------------
def split_pdf_bytes(pdf_bytes, pages_per_chunk=PAGES_PER_CHUNK):
    reader = PdfReader(io.BytesIO(pdf_bytes))
    total_pages = len(reader.pages)
    chunks = []

    for i in range(0, total_pages, pages_per_chunk):
        writer = PdfWriter()
        for page_num in range(i, min(i + pages_per_chunk, total_pages)):
            writer.add_page(reader.pages[page_num])
        chunk_stream = io.BytesIO()
        writer.write(chunk_stream)
        chunk_stream.seek(0)
        chunks.append(chunk_stream.read())
    return chunks

# --------------------------
# MAIN PROCESS
# --------------------------
def ocr_large_pdf_gcs():
    # ✅ Use project ID explicitly for storage client
    storage_client = storage.Client(project=PROJECT_ID)
    bucket_name, blob_name = GCS_INPUT_URI[5:].split("/", 1)
    blob = storage_client.bucket(bucket_name).blob(blob_name)
    pdf_bytes = blob.download_as_bytes()
    print(f"Downloaded PDF ({len(pdf_bytes)//1024} KB) from GCS.")

    # Split PDF into chunks in memory
    chunks = split_pdf_bytes(pdf_bytes)
    print(f"Split PDF into {len(chunks)} chunks ({PAGES_PER_CHUNK} pages per chunk).")

    # Process chunks in parallel
    combined_text = ""
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_pdf_chunk_bytes, c) for c in chunks]
        for f in futures:
            combined_text += f.result() + "\n"

    # Save final OCR
    store_document(OUTPUT_TXT_PATH, combined_text)
    print(f"All chunks processed. Final OCR saved at: {OUTPUT_TXT_PATH}")

# --------------------------
# RUN
# --------------------------
if __name__ == "__main__":
    ocr_large_pdf_gcs()


In [ ]:
# Only speed needs to be fixed.

import io
import time
from concurrent.futures import ThreadPoolExecutor
from google.cloud import documentai_v1 as documentai
from google.cloud import storage
from PyPDF2 import PdfReader, PdfWriter
import os

# --------------------------
# CONFIGURATION
# --------------------------
PROJECT_ID = "your-gcp-project-id"
LOCATION = "us"
PROCESSOR_ID = "your-processor-id"
GCS_INPUT_URI = "gs://your-bucket/7 March 1952 Badr.pdf"
OUTPUT_TXT_PATH = "output/7 March 1952 Badr.txt"

MAX_WORKERS = 6
PAGES_PER_CHUNK = 15  # Must not exceed processor page limit

# --------------------------
# UTILITIES
# --------------------------
def store_document(path: str, doc_content: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(doc_content)

def parse_documentai_doc(doc, start_page=0):
    """Extract text from Document AI document per page, with correct global page numbers."""
    parsed_pages = []
    for page in doc.pages:
        page_number = (page.page_number or 1) + start_page
        page_text = ""
        if hasattr(page, "layout") and page.layout and hasattr(page.layout, "text_anchor"):
            segments = page.layout.text_anchor.text_segments
            for seg in segments:
                start_idx = seg.start_index or 0
                end_idx = seg.end_index or len(doc.text)
                page_text += doc.text[start_idx:end_idx]
        if not page_text.strip():
            page_text = "[BLANK PAGE]"
        parsed_pages.append(f"\n--- PAGE {page_number} ---\n{page_text.strip()}\n--- END PAGE {page_number} ---\n")
    return "\n".join(parsed_pages)

def process_pdf_chunk_bytes(chunk_bytes, start_page=0):
    """Send PDF chunk bytes to Document AI using raw_document and apply page offset."""
    client_options = {"api_endpoint": f"{LOCATION}-documentai.googleapis.com"}
    client = documentai.DocumentProcessorServiceClient(client_options=client_options)
    processor_name = client.processor_path(PROJECT_ID, LOCATION, PROCESSOR_ID)

    raw_doc = documentai.RawDocument(content=chunk_bytes, mime_type="application/pdf")
    request = documentai.ProcessRequest(name=processor_name, raw_document=raw_doc)
    doc = client.process_document(request=request).document
    return parse_documentai_doc(doc, start_page=start_page)

# --------------------------
# SPLIT PDF IN MEMORY
# --------------------------
def split_pdf_bytes(pdf_bytes, pages_per_chunk=PAGES_PER_CHUNK):
    reader = PdfReader(io.BytesIO(pdf_bytes))
    total_pages = len(reader.pages)
    chunks = []

    for i in range(0, total_pages, pages_per_chunk):
        writer = PdfWriter()
        for page_num in range(i, min(i + pages_per_chunk, total_pages)):
            writer.add_page(reader.pages[page_num])
        chunk_stream = io.BytesIO()
        writer.write(chunk_stream)
        chunk_stream.seek(0)
        chunks.append(chunk_stream.read())
    return chunks

# --------------------------
# MAIN PROCESS
# --------------------------
def ocr_large_pdf_gcs():
    start_time = time.time()  # start timer

    # ✅ Use project ID explicitly for storage client
    storage_client = storage.Client(project=PROJECT_ID)
    bucket_name, blob_name = GCS_INPUT_URI[5:].split("/", 1)
    blob = storage_client.bucket(bucket_name).blob(blob_name)
    pdf_bytes = blob.download_as_bytes()
    print(f"Downloaded PDF ({len(pdf_bytes)//1024} KB) from GCS.")

    # Split PDF into chunks in memory
    chunks = split_pdf_bytes(pdf_bytes)
    print(f"Split PDF into {len(chunks)} chunks ({PAGES_PER_CHUNK} pages per chunk).")

    # Process chunks in parallel
    combined_text = ""
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = []
        for i, c in enumerate(chunks):
            start_page = i * PAGES_PER_CHUNK
            futures.append(executor.submit(process_pdf_chunk_bytes, c, start_page=start_page))
        for f in futures:
            combined_text += f.result() + "\n"

    # Save final OCR
    store_document(OUTPUT_TXT_PATH, combined_text)
    end_time = time.time()  # end timer
    print(f"All chunks processed. Final OCR saved at: {OUTPUT_TXT_PATH}")
    print(f"Total OCR processing time: {end_time - start_time:.2f} seconds")

# --------------------------
# RUN
# --------------------------
if __name__ == "__main__":
    ocr_large_pdf_gcs()


In [ ]:
"""
Document AI OCR with Best Practices
====================================

This implementation follows Google Cloud Document AI best practices:

1. OCR Quality Optimization:
   - Language hints for Urdu/Arabic/English
   - Native PDF parsing enabled
   - Pretty print formatting
   
2. Error Handling:
   - Exponential backoff with jitter
   - Retryable vs non-retryable error detection
   - Proper error logging
   
3. Performance:
   - Optimal worker count (2x CPU cores, capped for rate limits)
   - Parallel processing with ThreadPoolExecutor
   - Connection reuse
   
4. Text Extraction:
   - Multiple extraction methods (blocks → paragraphs → lines)
   - Preserves RTL formatting for Urdu/Arabic
   - Maintains proper page numbering
   
5. Monitoring:
   - Progress tracking
   - Success rate reporting
   - Performance metrics
"""

import io
import os
import time
import random
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.cloud import documentai_v1 as documentai
from google.cloud import storage
from google.api_core.exceptions import GoogleAPICallError, RetryError
from google.api_core.retry import Retry
from PyPDF2 import PdfReader, PdfWriter
from collections import OrderedDict

# --------------------------
# CONFIGURATION - Best Practices
# --------------------------
PROJECT_ID = "your-gcp-project-id"
LOCATION = "us"
PROCESSOR_ID = "your-processor-id"
GCS_INPUT_URI = "gs://your-bucket/sahirat-al-tarf-fi-al-istiadhah-wa-al-basmalah-wa-al-ism-wa-al-fil-wa-al-harf-TTUrDNtVsf.pdf"
OUTPUT_TXT_PATH = "output/al-ashbah-wa-l-naza'ir-fi-al-nahw-18aEYaH_W6.txt"

# Performance Settings (Best Practices)
MAX_WORKERS = 10          # Optimal: 2x CPU cores, but capped for API rate limits
PAGES_PER_CHUNK = 15      # Max pages per chunk (Document AI limit is 15 in non-imageless mode)
MAX_RETRIES = 5           # Best practice: 3-5 retries for transient errors
REQUEST_TIMEOUT = 900     # Best practice: 15 min timeout for large documents
INITIAL_RETRY_DELAY = 1   # Initial retry delay in seconds
MAX_RETRY_DELAY = 60      # Maximum retry delay in seconds
RETRY_MULTIPLIER = 2      # Exponential backoff multiplier

# OCR Quality Settings (Best Practices)
OCR_LANGUAGE_HINTS = ["ur", "ar", "en"]  # Language hints improve accuracy significantly
ENABLE_NATIVE_PDF_PARSING = True         # Best practice: Use native PDF parsing
ENABLE_PRETTY_PRINT = True               # Best practice: Enable pretty print
ENABLE_IMAGE_QUALITY_SCORES = True        # Get quality scores for debugging

# Logging Configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --------------------------
# UTILITIES
# --------------------------
def store_document(path: str, doc_content: str):
    """Save text to a local file."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(doc_content)

def extract_text_from_layout(layout, doc_text):
    """Extract text from a layout element (best practice: reusable helper)."""
    if not hasattr(layout, "text_anchor") or not layout.text_anchor:
        return None
    
    segments = layout.text_anchor.text_segments
    if not segments:
        return None
    
    valid_starts = [seg.start_index for seg in segments if seg.start_index is not None]
    valid_ends = [seg.end_index for seg in segments if seg.end_index is not None]
    
    if not valid_starts or not valid_ends:
        return None
    
    return doc_text[min(valid_starts):max(valid_ends)]

def parse_documentai_doc(doc, start_page=1):
    """
    Extract plain text per page from Document AI response with best practices:
    - Uses structured elements (blocks/paragraphs/lines) for better extraction
    - Preserves natural line breaks and RTL formatting for Urdu/Arabic
    - Handles edge cases gracefully
    - Maintains correct global page numbering
    
    Note: Document AI returns page.page_number relative to the chunk (1-N), not global.
    We ignore page.page_number and use enumerate with start_page offset instead.
    """
    parsed_pages = []
    # Use enumerate to get sequential page numbers starting from start_page
    # Document AI's page.page_number is relative to chunk, so we ignore it
    for idx, page in enumerate(doc.pages):
        global_page_num = start_page + idx  # Calculate correct global page number
        page_text = ""
        
        # Best practice: Try multiple extraction methods in order of preference
        # Method 0: Use full document text first (often best formatted)
        if doc.text and hasattr(page, "layout") and page.layout:
            page_text = extract_text_from_layout(page.layout, doc.text)
        
        # Method 1: Try blocks (best structure preservation)
        if not page_text and hasattr(page, "blocks") and page.blocks:
            block_texts = []
            for block in page.blocks:
                if hasattr(block, "layout") and block.layout:
                    block_text = extract_text_from_layout(block.layout, doc.text)
                    if block_text and block_text.strip():
                        block_texts.append(block_text)
            if block_texts:
                page_text = "\n\n".join(block_texts)  # Double newline for paragraph separation
        
        # Method 2: Try paragraphs
        if not page_text and hasattr(page, "paragraphs") and page.paragraphs:
            para_texts = []
            for para in page.paragraphs:
                if hasattr(para, "layout") and para.layout:
                    para_text = extract_text_from_layout(para.layout, doc.text)
                    if para_text and para_text.strip():
                        para_texts.append(para_text)
            if para_texts:
                page_text = "\n".join(para_texts)
        
        # Method 3: Try lines (line-by-line extraction)
        if not page_text and hasattr(page, "lines") and page.lines:
            line_texts = []
            for line in page.lines:
                if hasattr(line, "layout") and line.layout:
                    line_text = extract_text_from_layout(line.layout, doc.text)
                    if line_text and line_text.strip():
                        line_texts.append(line_text)
            if line_texts:
                page_text = "\n".join(line_texts)
        
        # Method 5: Last resort - estimate page boundaries
        if not page_text.strip() and doc.text:
            total_pages = len(doc.pages)
            if total_pages > 0:
                chars_per_page = len(doc.text) // total_pages
                start_char = idx * chars_per_page
                end_char = (idx + 1) * chars_per_page if idx < total_pages - 1 else len(doc.text)
                page_text = doc.text[start_char:end_char]
        
        if not page_text.strip():
            page_text = "[BLANK PAGE]"
        else:
            # Clean up: preserve structure but remove excessive blank lines
            lines = page_text.split('\n')
            cleaned_lines = []
            prev_empty = False
            
            for line in lines:
                # Preserve line structure for RTL text
                if line.strip():  # Non-empty line
                    cleaned_lines.append(line)
                    prev_empty = False
                elif not prev_empty:
                    # Keep one blank line between paragraphs
                    cleaned_lines.append('')
                    prev_empty = True
            
            # Remove trailing blank lines
            while cleaned_lines and not cleaned_lines[-1]:
                cleaned_lines.pop()
            
            page_text = '\n'.join(cleaned_lines)
        
        parsed_pages.append(f"\n--- PAGE {global_page_num} ---\n{page_text}\n--- END PAGE {global_page_num} ---\n")
    return "\n".join(parsed_pages)

# --------------------------
# DOCUMENT AI CALL
# --------------------------
def calculate_retry_delay(attempt, base_delay=INITIAL_RETRY_DELAY, max_delay=MAX_RETRY_DELAY, multiplier=RETRY_MULTIPLIER):
    """Calculate exponential backoff delay with jitter (best practice)."""
    delay = min(base_delay * (multiplier ** (attempt - 1)), max_delay)
    # Add jitter to prevent thundering herd problem
    jitter = random.uniform(0, delay * 0.1)
    return delay + jitter

def is_retryable_error(error):
    """Determine if an error is retryable (best practice)."""
    error_str = str(error).lower()
    # Non-retryable errors
    if any(term in error_str for term in ["page_limit_exceeded", "invalid_argument", "permission_denied", "not_found"]):
        return False
    # Retryable errors
    if any(term in error_str for term in ["deadline_exceeded", "unavailable", "internal", "resource_exhausted", "aborted"]):
        return True
    return True  # Default to retryable for unknown errors

def process_pdf_chunk_bytes(chunk_bytes, start_page=1, chunk_id=0):
    """
    Send a PDF chunk to Document AI with best practices:
    - Exponential backoff with jitter
    - Proper error handling
    - OCR quality optimization
    - Connection reuse
    """
    # Best practice: Reuse client connection (create once per thread)
    client_options = {"api_endpoint": f"{LOCATION}-documentai.googleapis.com"}
    client = documentai.DocumentProcessorServiceClient(client_options=client_options)
    processor_name = client.processor_path(PROJECT_ID, LOCATION, PROCESSOR_ID)

    raw_doc = documentai.RawDocument(content=chunk_bytes, mime_type="application/pdf")
    
    # Best practice: Build request with OCR quality options
    request_kwargs = {
        "name": processor_name,
        "raw_document": raw_doc,
    }
    
    # Add OCR configuration if available (best practice for quality)
    try:
        if hasattr(documentai, 'ProcessOptions') and hasattr(documentai, 'OcrConfig'):
            ocr_config = documentai.OcrConfig(
                language_hints=OCR_LANGUAGE_HINTS,
                enable_native_pdf_parsing=ENABLE_NATIVE_PDF_PARSING,
                enable_pretty_print=ENABLE_PRETTY_PRINT,
            )
            request_kwargs["process_options"] = documentai.ProcessOptions(ocr_config=ocr_config)
    except Exception as e:
        logger.warning(f"Could not set OCR config: {e}. Using default settings.")
    
    request = documentai.ProcessRequest(**request_kwargs)

    # Best practice: Retry with exponential backoff and jitter
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            logger.info(f"Processing chunk {chunk_id} (page {start_page}), attempt {attempt}/{MAX_RETRIES}")
            doc = client.process_document(request=request, timeout=REQUEST_TIMEOUT).document
            
            # Best practice: Validate response
            if not doc or not doc.text:
                raise ValueError("Empty document response from Document AI")
            
            logger.info(f"✅ Successfully processed chunk {chunk_id} (page {start_page})")
            return parse_documentai_doc(doc, start_page=start_page), chunk_id

        except GoogleAPICallError as e:
            last_error = e
            error_str = str(e)
            
            # Best practice: Don't retry non-retryable errors
            if not is_retryable_error(e):
                logger.error(f"❌ Non-retryable error for chunk {chunk_id} (page {start_page}): {e}")
                if "PAGE_LIMIT_EXCEEDED" in error_str or "exceed the limit" in error_str:
                    return f"\n--- PAGE {start_page} ---\n[FAILED TO OCR - PAGE LIMIT EXCEEDED]\n", chunk_id
                return f"\n--- PAGE {start_page} ---\n[FAILED TO OCR - {error_str[:100]}]\n", chunk_id
            
            # Best practice: Exponential backoff with jitter
            if attempt < MAX_RETRIES:
                wait_time = calculate_retry_delay(attempt)
                logger.warning(f"⚠️ Chunk {chunk_id} (page {start_page}) failed (attempt {attempt}/{MAX_RETRIES}): {e}")
                logger.info(f"   Retrying in {wait_time:.2f} seconds...")
                time.sleep(wait_time)
            else:
                logger.error(f"❌ Chunk {chunk_id} (page {start_page}) failed after {MAX_RETRIES} retries: {e}")
                return f"\n--- PAGE {start_page} ---\n[FAILED TO OCR AFTER {MAX_RETRIES} RETRIES]\n", chunk_id
                
        except Exception as e:
            last_error = e
            if attempt < MAX_RETRIES:
                wait_time = calculate_retry_delay(attempt)
                logger.warning(f"⚠️ Chunk {chunk_id} (page {start_page}) error (attempt {attempt}/{MAX_RETRIES}): {e}")
                logger.info(f"   Retrying in {wait_time:.2f} seconds...")
                time.sleep(wait_time)
            else:
                logger.error(f"❌ Chunk {chunk_id} (page {start_page}) failed after {MAX_RETRIES} retries: {e}")
                return f"\n--- PAGE {start_page} ---\n[FAILED TO OCR - {str(e)[:100]}]\n", chunk_id
    
    # Should not reach here, but handle it
    return f"\n--- PAGE {start_page} ---\n[FAILED TO OCR]\n", chunk_id

# --------------------------
# SPLIT PDF INTO CHUNKS
# --------------------------
def split_pdf_bytes(pdf_bytes, pages_per_chunk=PAGES_PER_CHUNK):
    reader = PdfReader(io.BytesIO(pdf_bytes))
    total_pages = len(reader.pages)
    chunks = []

    for i in range(0, total_pages, pages_per_chunk):
        writer = PdfWriter()
        for page_num in range(i, min(i + pages_per_chunk, total_pages)):
            writer.add_page(reader.pages[page_num])
        chunk_stream = io.BytesIO()
        writer.write(chunk_stream)
        chunk_stream.seek(0)
        # Return (chunk_bytes, start_page, chunk_id) for proper ordering
        chunks.append((chunk_stream.read(), i + 1, i // pages_per_chunk))
    return chunks

# --------------------------
# MAIN PROCESS
# --------------------------
def ocr_large_pdf_gcs():
    """
    Main OCR processing function with best practices:
    - Proper error handling
    - Progress tracking
    - Resource management
    - Performance monitoring
    """
    start_time = time.time()
    logger.info("=" * 60)
    logger.info("Starting OCR processing with best practices")
    logger.info("=" * 60)

    try:
        # Best practice: Download from GCS with error handling
        logger.info(f"Downloading PDF from {GCS_INPUT_URI}")
        storage_client = storage.Client(project=PROJECT_ID)
        bucket_name, blob_name = GCS_INPUT_URI[5:].split("/", 1)
        blob = storage_client.bucket(bucket_name).blob(blob_name)
        pdf_bytes = blob.download_as_bytes()
        logger.info(f"✅ Downloaded PDF ({len(pdf_bytes)//1024} KB) from GCS.")

        # Best practice: Split into chunks
        chunks = split_pdf_bytes(pdf_bytes)
        logger.info(f"📄 Split PDF into {len(chunks)} chunks ({PAGES_PER_CHUNK} pages per chunk).")
        logger.info(f"🚀 Processing with {MAX_WORKERS} parallel workers...\n")

        # Best practice: Use OrderedDict to maintain chunk order
        results = OrderedDict()
        completed_count = 0
        failed_count = 0
        chunk_start_times = {}

        # Best practice: Process chunks in parallel with proper resource management
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            # Submit all tasks
            futures = {
                executor.submit(process_pdf_chunk_bytes, c[0], c[1], c[2]): c[2] 
                for c in chunks
            }
            chunk_start_times = {futures[f]: time.time() for f in futures}
            
            # Process results as they complete (best practice: handle as they finish)
            for future in as_completed(futures):
                chunk_id = futures[future]
                chunk_time = time.time() - chunk_start_times[chunk_id]
                try:
                    text, _ = future.result()
                    results[chunk_id] = text
                    completed_count += 1
                    logger.info(f"✅ Chunk {chunk_id + 1}/{len(chunks)} completed in {chunk_time:.2f}s ({completed_count}/{len(chunks)} total)")
                except Exception as e:
                    failed_count += 1
                    logger.error(f"❌ Chunk {chunk_id + 1} failed after {chunk_time:.2f}s: {e}")
                    results[chunk_id] = f"\n--- PAGE {chunks[chunk_id][1]} ---\n[ERROR PROCESSING CHUNK]\n"

        # Best practice: Combine results in order
        combined_text = "\n".join([results[i] for i in sorted(results.keys())])

        # Best practice: Save with error handling
        store_document(OUTPUT_TXT_PATH, combined_text)
        
        elapsed = time.time() - start_time
        logger.info("=" * 60)
        logger.info(f"✅ OCR processing complete!")
        logger.info(f"   Output: {OUTPUT_TXT_PATH}")
        logger.info(f"   Total time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")
        logger.info(f"   Average per chunk: {elapsed/len(chunks):.2f} seconds")
        logger.info(f"   Success rate: {completed_count}/{len(chunks)} chunks ({100*completed_count/len(chunks):.1f}%)")
        if failed_count > 0:
            logger.warning(f"   Failed chunks: {failed_count}")
        logger.info("=" * 60)
        
    except Exception as e:
        logger.error(f"❌ Fatal error in OCR processing: {e}", exc_info=True)
        raise

# --------------------------
# HELPER: Re-process failed chunks from previous run
# --------------------------
def reprocess_failed_chunks(failed_pages_list):
    """Re-process specific pages that failed in a previous OCR run.
    
    Args:
        failed_pages_list: List of page numbers that failed (e.g., [1, 21, 41, 61])
    """
    start_time = time.time()
    
    # Download from GCS
    storage_client = storage.Client(project=PROJECT_ID)
    bucket_name, blob_name = GCS_INPUT_URI[5:].split("/", 1)
    blob = storage_client.bucket(bucket_name).blob(blob_name)
    pdf_bytes = blob.download_as_bytes()
    
    reader = PdfReader(io.BytesIO(pdf_bytes))
    total_pages = len(reader.pages)
    
    # Group failed pages into chunks
    failed_chunks = {}
    for page_num in failed_pages_list:
        if 1 <= page_num <= total_pages:
            chunk_start = ((page_num - 1) // PAGES_PER_CHUNK) * PAGES_PER_CHUNK + 1
            if chunk_start not in failed_chunks:
                failed_chunks[chunk_start] = []
            failed_chunks[chunk_start].append(page_num)
    
    print(f"📄 Re-processing {len(failed_chunks)} failed chunks...")
    
    results = {}
    for chunk_start in sorted(failed_chunks.keys()):
        # Create chunk
        chunk_end = min(chunk_start + PAGES_PER_CHUNK - 1, total_pages)
        writer = PdfWriter()
        for page_num in range(chunk_start - 1, chunk_end):
            writer.add_page(reader.pages[page_num])
        chunk_stream = io.BytesIO()
        writer.write(chunk_stream)
        chunk_stream.seek(0)
        chunk_bytes = chunk_stream.read()
        
        chunk_id = (chunk_start - 1) // PAGES_PER_CHUNK
        text, _ = process_pdf_chunk_bytes(chunk_bytes, chunk_start, chunk_id)
        results[chunk_id] = text
        print(f"✅ Re-processed chunk starting at page {chunk_start}")
    
    # Read existing file and replace failed chunks
    if os.path.exists(OUTPUT_TXT_PATH):
        with open(OUTPUT_TXT_PATH, "r", encoding="utf-8") as f:
            existing_text = f.read()
        
        # Replace failed chunks with new results
        for chunk_id, new_text in results.items():
            # Find and replace the failed chunk section
            pattern = f"--- PAGE {chunk_id * PAGES_PER_CHUNK + 1} ---"
            if pattern in existing_text:
                # Extract the chunk section and replace
                start_idx = existing_text.find(pattern)
                next_chunk_start = existing_text.find("--- PAGE ", start_idx + len(pattern))
                if next_chunk_start == -1:
                    next_chunk_start = len(existing_text)
                existing_text = existing_text[:start_idx] + new_text + existing_text[next_chunk_start:]
        
        store_document(OUTPUT_TXT_PATH, existing_text)
    else:
        # If file doesn't exist, just save the results
        combined_text = "\n".join([results[i] for i in sorted(results.keys())])
        store_document(OUTPUT_TXT_PATH, combined_text)
    
    elapsed = time.time() - start_time
    print(f"\n✅ Re-processing complete. Updated: {OUTPUT_TXT_PATH}")
    print(f"⏱️ Time: {elapsed:.2f} seconds")

# --------------------------
# RUN
# --------------------------
if __name__ == "__main__":
    ocr_large_pdf_gcs()
    
    # Uncomment below to re-process only failed pages from previous run:
    # reprocess_failed_chunks([1, 21, 41, 61, 81, 101, 121, 141, 161, 181, 201, 221, 241])


In [ ]:
"""
Hybrid OCR Approach: Document AI + Gemini Vision API
====================================================

This implementation uses:
1. Document AI as primary OCR (fast, cost-effective)
2. Gemini Vision API as fallback for difficult pages (better accuracy)
3. Quality scoring to determine which pages need Gemini
4. Intelligent result combination

Best for: Maximum accuracy with cost optimization
"""

import io
import os
import time
import base64
import random
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.cloud import documentai_v1 as documentai
from google.cloud import storage
from google.api_core.exceptions import GoogleAPICallError
from PyPDF2 import PdfReader, PdfWriter
from collections import OrderedDict
import google.generativeai as genai
from PIL import Image
import pdf2image
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.docstore.document import Document

# Load environment variables from .env file
load_dotenv()

# --------------------------
# CONFIGURATION - Hybrid Approach
# Google Cloud: use service account JSON (your-gcp-project-id.json)
# --------------------------

import json

# Path to the service account JSON (same folder as notebook)
_CREDENTIALS_JSON_PATH = "your-gcp-project-id.json"
if not os.path.isfile(_CREDENTIALS_JSON_PATH):
    _CREDENTIALS_JSON_PATH = os.path.join(os.path.expanduser("~"), "Desktop", "venv", "your-gcp-project-id.json")
if os.path.isfile(_CREDENTIALS_JSON_PATH):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.path.abspath(_CREDENTIALS_JSON_PATH)
    with open(_CREDENTIALS_JSON_PATH, "r") as f:
        _sa = json.load(f)
    PROJECT_ID = _sa.get("project_id", "your-gcp-project-id")
else:
    PROJECT_ID = os.getenv("PROJECT_ID", "your-gcp-project-id")

LOCATION = "us"
# Processor must be in project your-gcp-project-id. Create in Console: Document AI > Processors > Document OCR.
PROCESSOR_ID = "your-processor-id"
GCS_INPUT_URI = "gs://your-bucket/al-tajrid-wa-al-ihtimam-bi-jam-fatawa-al-walid-shaykh-al-islam-Wane5cZhGL.pdf"
OUTPUT_TXT_PATH = "output/Ilm_al-Din_al-Balqini_ocr/al-tajrid-wa-al-ihtimam-bi-jam-fatawa-al-walid-shaykh-al-islam.txt"

# Author configuration - set this to the author name for Chroma database
# This will be used as the persist_directory name (e.g., "./chroma_db/AUTHOR_NAME")
AUTHOR_NAME = "Ilm_al-Din_al-Balqini_ocr"  # Change this to your author name

# Document AI Settings
MAX_WORKERS = 10
PAGES_PER_CHUNK = 15
MAX_RETRIES = 8
REQUEST_TIMEOUT = 900

# Gemini Vision API Settings
GEMINI_API_KEY = os.environ["GOOGLE_API_KEY"]  # Set your API key in .env file
USE_GEMINI_FOR_LOW_QUALITY = True  # Use Gemini for pages with low confidence
QUALITY_THRESHOLD = 0.7  # Confidence threshold (0-1) below which to use Gemini
GEMINI_MODEL = "gemini-2.5-pro"  # Latest Gemini model with improved vision capabilities

# Chroma Embedding Settings
EMBEDDING_BATCH_SIZE = 20  # Batch size for embeddings (Chroma works well with smaller batches)
EMBEDDING_DELAY = 0.2  # Delay in seconds between batch embedding calls (to respect rate limits)
EMBEDDING_MAX_RETRIES = 6  # Maximum retries for embedding failures
EMBEDDING_RETRY_DELAY = 2  # Initial retry delay in seconds (exponential backoff)

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Initialize Gemini (with error handling for gemini-2.5-pro)
gemini_model = None
if GEMINI_API_KEY:
    try:
        genai.configure(api_key=GEMINI_API_KEY)
        # Try to initialize the model
        gemini_model = genai.GenerativeModel(GEMINI_MODEL)
        # Test if model is accessible
        logger.info(f"✅ Gemini Vision API initialized with model: {GEMINI_MODEL}")
    except Exception as e:
        logger.error(f"❌ Failed to initialize Gemini model {GEMINI_MODEL}: {e}")
        logger.error(f"   Error type: {type(e).__name__}")
        # Try alternative model names if gemini-2.5-pro fails
        alternative_models = ["gemini-2.0-pro", "gemini-1.5-pro", "gemini-1.5-flash"]
        for alt_model in alternative_models:
            try:
                logger.info(f"   Trying alternative model: {alt_model}")
                gemini_model = genai.GenerativeModel(alt_model)
                logger.info(f"✅ Successfully initialized with {alt_model}")
                GEMINI_MODEL = alt_model  # Update the model name
                break
            except Exception as alt_error:
                logger.warning(f"   {alt_model} also failed: {alt_error}")
                continue
        if not gemini_model:
            logger.warning("⚠️ Could not initialize any Gemini model. Will use Document AI only.")
else:
    logger.warning("⚠️ Gemini API key not set. Will use Document AI only.")

# --------------------------
# CHROMA VECTOR STORE INITIALIZATION
# --------------------------
# Set GOOGLE_API_KEY for embeddings (required for Generative Language API)
# Use GEMINI_API_KEY if GOOGLE_API_KEY is not set
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", os.getenv("GEMINI_API_KEY", ""))
if GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

# Initialize Chroma vectorstore for embeddings
vectorstore = None
try:
    # Initialize Gemini embeddings with explicit API key
    # GoogleGenerativeAIEmbeddings uses GOOGLE_API_KEY from environment
    if not GOOGLE_API_KEY:
        logger.warning("⚠️ GOOGLE_API_KEY not set. Embeddings may fail due to insufficient authentication scopes.")
        logger.warning("   Set GOOGLE_API_KEY in .env file or use GEMINI_API_KEY")
        logger.warning("   Attempting to use GEMINI_API_KEY as fallback...")
        if GEMINI_API_KEY:
            GOOGLE_API_KEY = GEMINI_API_KEY
            os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
            logger.info("   Using GEMINI_API_KEY as GOOGLE_API_KEY")
    
    if not GOOGLE_API_KEY:
        logger.error("❌ No API key available for embeddings. Cannot initialize Chroma.")
        logger.error("   Please set either GOOGLE_API_KEY or GEMINI_API_KEY in environment or .env file")
        vectorstore = None
    else:
        logger.info(f"🔑 Using GOOGLE_API_KEY for embeddings (key length: {len(GOOGLE_API_KEY)} chars)")
        gemini_embeddings = GoogleGenerativeAIEmbeddings(
            model="models/gemini-embedding-001",
            google_api_key=GOOGLE_API_KEY
        )
        
        # Create Chroma persist directory based on author name
        chroma_persist_dir = f"./chroma_db/{AUTHOR_NAME}"
        os.makedirs(chroma_persist_dir, exist_ok=True)
        logger.info(f"📁 Chroma persist directory: {chroma_persist_dir}")
        
        # Create Chroma vectorstore
        vectorstore = Chroma(
            persist_directory=chroma_persist_dir,
            embedding_function=gemini_embeddings
        )
        logger.info(f"✅ Chroma vectorstore initialized (persist_directory: {chroma_persist_dir}, author: {AUTHOR_NAME})")
        # Batch embedding manager will be initialized lazily when first page is embedded
except Exception as e:
    logger.error(f"❌ Failed to initialize Chroma: {e}")
    logger.error(f"   Error type: {type(e).__name__}")
    import traceback
    logger.error(f"   Traceback: {traceback.format_exc()}")
    if "ACCESS_TOKEN_SCOPE_INSUFFICIENT" in str(e) or "403" in str(e):
        logger.error("   This is an authentication scope error.")
        logger.error("   Solution: Set GOOGLE_API_KEY in .env file (can use same value as GEMINI_API_KEY)")
    vectorstore = None

# --------------------------
# UTILITIES
# --------------------------
def store_document(path: str, doc_content: str):
    """Save text to a local file."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(doc_content)

def pdf_page_to_image(pdf_bytes, page_num):
    """Convert a single PDF page to PIL Image."""
    try:
        images = pdf2image.convert_from_bytes(pdf_bytes, first_page=page_num+1, last_page=page_num+1, dpi=300)
        return images[0] if images else None
    except Exception as e:
        logger.error(f"Error converting page {page_num} to image: {e}")
        return None

def image_to_base64(image):
    """Convert PIL Image to base64 string."""
    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode()

# --------------------------
# BATCH EMBEDDING MANAGER
# --------------------------
class BatchEmbeddingManager:
    """Manages batch embedding to improve performance and respect rate limits."""
    def __init__(self, batch_size=EMBEDDING_BATCH_SIZE):
        self.batch_size = batch_size
        self.batch = []
        self.embedded_count = 0
        self.failed_count = 0
    
    def add_page(self, page_text, page_num, source_file=None):
        """Add a page to the batch. Embeds automatically when batch is full."""
        global vectorstore
        if vectorstore is None:
            logger.warning(f"⚠️ Vectorstore not available, skipping embedding for page {page_num}")
            return False
        if not page_text or not page_text.strip():
            logger.warning(f"⚠️ Empty page text, skipping embedding for page {page_num}")
            return False
        
        # Extract book name
        if source_file:
            book_name = source_file.replace(".pdf", "").replace(".PDF", "")
        else:
            filename = GCS_INPUT_URI.split("/")[-1]
            book_name = filename.replace(".pdf", "").replace(".PDF", "")
        
        # Create Document with author metadata (like Chroma RAG notebook)
        doc = Document(
            page_content=page_text,
            metadata={
                "book": book_name,  # Use "book" to match Chroma RAG notebook format
                "page": str(page_num),  # Convert to string to match Chroma RAG format
                "author": AUTHOR_NAME,  # Add author metadata
                "bookname": book_name,  # Keep for backward compatibility
                "page_number": page_num,
                "source_file": source_file or GCS_INPUT_URI.split("/")[-1],
                "method": "hybrid_ocr"
            }
        )
        
        self.batch.append(doc)
        logger.debug(f"📝 Added page {page_num} to embedding batch (batch size: {len(self.batch)}/{self.batch_size})")
        
        # Embed batch when it reaches batch_size
        if len(self.batch) >= self.batch_size:
            logger.info(f"📦 Batch full ({len(self.batch)} pages), embedding now...")
            return self._embed_batch()
        
        return True
    
    def _embed_batch(self, max_retries=None, retry_delay=None):
        """Embed the current batch with retry logic."""
        global vectorstore
        if not self.batch:
            return True
        
        if max_retries is None:
            max_retries = EMBEDDING_MAX_RETRIES
        if retry_delay is None:
            retry_delay = EMBEDDING_RETRY_DELAY
        
        batch_to_embed = self.batch.copy()
        self.batch = []  # Clear batch
        
        for attempt in range(1, max_retries + 1):
            try:
                # Add delay between batch embedding calls
                if attempt > 1:
                    delay = retry_delay * (2 ** (attempt - 2))
                    logger.info(f"   Retrying batch embedding (attempt {attempt}/{max_retries}) after {delay}s...")
                    time.sleep(delay)
                elif hasattr(BatchEmbeddingManager, '_last_batch_time'):
                    elapsed = time.time() - BatchEmbeddingManager._last_batch_time
                    if elapsed < EMBEDDING_DELAY:
                        time.sleep(EMBEDDING_DELAY - elapsed)
                
                # Embed batch
                logger.debug(f"🔄 Calling vectorstore.add_documents with {len(batch_to_embed)} documents...")
                vectorstore.add_documents(batch_to_embed)
                logger.debug("✅ add_documents completed successfully")
                
                # Persist Chroma database after each batch
                vectorstore.persist()
                logger.debug("✅ Chroma database persisted")
                
                BatchEmbeddingManager._last_batch_time = time.time()
                
                self.embedded_count += len(batch_to_embed)
                logger.info(f"✅ Embedded batch of {len(batch_to_embed)} pages in Chroma (total: {self.embedded_count} pages)")
                return True
                
            except Exception as e:
                error_str = str(e)
                
                # Check for rate limit errors (429)
                if "429" in error_str or "quota" in error_str.lower() or "rate limit" in error_str.lower():
                    if attempt < max_retries:
                        delay = retry_delay * (2 ** (attempt - 1))
                        if not hasattr(BatchEmbeddingManager, '_rate_limit_logged'):
                            logger.warning(f"⚠️ Rate limit hit for batch embeddings. Retrying with exponential backoff...")
                            BatchEmbeddingManager._rate_limit_logged = True
                        time.sleep(delay)
                        continue
                    else:
                        if not hasattr(BatchEmbeddingManager, '_rate_limit_final_logged'):
                            logger.error(f"❌ Rate limit exceeded for batch embeddings after {max_retries} retries")
                            logger.error(f"   {len(batch_to_embed)} pages failed to embed")
                            BatchEmbeddingManager._rate_limit_final_logged = True
                        self.failed_count += len(batch_to_embed)
                        return False
                
                # Other errors
                else:
                    if attempt < max_retries:
                        logger.warning(f"⚠️ Batch embedding failed (attempt {attempt}/{max_retries}): {e}")
                        continue
                    else:
                        logger.error(f"❌ Batch embedding failed after {max_retries} attempts: {e}")
                        self.failed_count += len(batch_to_embed)
                        return False
        
        return False
    
    def flush(self):
        """Embed any remaining pages in the batch."""
        global vectorstore
        if self.batch:
            logger.info(f"🔄 Flushing {len(self.batch)} remaining pages in batch...")
            result = self._embed_batch()
            # Persist Chroma database after flushing
            if vectorstore is not None:
                vectorstore.persist()
                logger.info("💾 Chroma database persisted after flush")
            return result
        else:
            logger.info("✅ No pages remaining in batch to flush")
        return True
    
    def get_stats(self):
        """Get embedding statistics."""
        return {
            "embedded": self.embedded_count,
            "failed": self.failed_count,
            "pending": len(self.batch)
        }

# Global batch embedding manager
embedding_manager = None

def initialize_embedding_manager():
    """Initialize the global batch embedding manager."""
    global embedding_manager, vectorstore
    if vectorstore is not None:
        embedding_manager = BatchEmbeddingManager(batch_size=EMBEDDING_BATCH_SIZE)
        logger.info(f"✅ Batch embedding manager initialized (batch_size: {EMBEDDING_BATCH_SIZE})")
    else:
        logger.warning("⚠️ Cannot initialize embedding manager: vectorstore is None")
    return embedding_manager

def embed_page_in_chroma(page_text, page_num, source_file=None):
    """
    Add a page to the batch embedding queue.
    Pages are embedded in batches for better performance and rate limit management.
    """
    global embedding_manager, vectorstore
    
    # Check if vectorstore is available
    if vectorstore is None:
        logger.warning(f"⚠️ Chroma vectorstore not initialized, cannot embed page {page_num}")
        return False
    
    # Initialize manager if not already done
    if embedding_manager is None:
        initialize_embedding_manager()
    
    # Use batch manager if available
    if embedding_manager:
        result = embedding_manager.add_page(page_text, page_num, source_file)
        if not result:
            logger.warning(f"⚠️ Failed to add page {page_num} to embedding batch")
        return result
    else:
        logger.warning(f"⚠️ Embedding manager not initialized, cannot embed page {page_num}")
        return False

def ocr_with_gemini_vision(image, page_num):
    """OCR a page using Gemini Vision API (compatible with gemini-2.5-pro)."""
    if not gemini_model:
        return None, 0.0
    
    try:
        # Prepare prompt for Urdu/Arabic OCR
        prompt = """Extract all text from this image. 
The text is in Urdu/Arabic (right-to-left) and may also contain English.
Preserve the exact formatting, line breaks, and alignment.
Output the text exactly as it appears, maintaining proper RTL structure.
Do not translate or summarize, just extract the text."""
        
        # Configure generation settings for better OCR (compatible with gemini-2.5-pro)
        try:
            # Try with generation config (newer API)
            from google.generativeai.types import HarmCategory, HarmBlockThreshold
            
            generation_config = genai.types.GenerationConfig(
                temperature=0.1,  # Low temperature for accurate extraction
                top_p=0.95,
                top_k=40,
                max_output_tokens=8192,  # Enough for full page text
            )
            
            # Safety settings - allow all content for OCR
            safety_settings = {
                HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
                HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
            }
            
            # Send to Gemini with proper configuration
            response = gemini_model.generate_content(
                [prompt, image],
                generation_config=generation_config,
                safety_settings=safety_settings
            )
        except (ImportError, AttributeError) as config_error:
            # Fallback: try with dict format or without config
            try:
                generation_config = {
                    "temperature": 0.1,
                    "top_p": 0.95,
                    "top_k": 40,
                    "max_output_tokens": 8192,
                }
                response = gemini_model.generate_content(
                    [prompt, image],
                    generation_config=generation_config
                )
            except Exception as dict_error:
                # Final fallback: try without config
                logger.warning(f"⚠️ Config error, trying without config: {config_error}")
                response = gemini_model.generate_content([prompt, image])
        
        # Handle response (compatible with different API versions including gemini-2.5-pro)
        if response:
            # Try different ways to get text based on API version
            text = None
            
            # Method 1: Direct text attribute (most common)
            if hasattr(response, 'text') and response.text:
                text = response.text.strip()
            
            # Method 2: Candidates structure (older API)
            elif hasattr(response, 'candidates') and response.candidates and len(response.candidates) > 0:
                candidate = response.candidates[0]
                # Try content.parts structure
                if hasattr(candidate, 'content') and hasattr(candidate.content, 'parts'):
                    if candidate.content.parts and len(candidate.content.parts) > 0:
                        if hasattr(candidate.content.parts[0], 'text'):
                            text = candidate.content.parts[0].text.strip()
                # Try direct text attribute
                elif hasattr(candidate, 'text'):
                    text = candidate.text.strip()
                # Try output field
                elif hasattr(candidate, 'output'):
                    text = str(candidate.output).strip()
            
            # Method 3: Check for blocked or filtered responses
            if hasattr(response, 'prompt_feedback'):
                feedback = response.prompt_feedback
                if hasattr(feedback, 'block_reason') and feedback.block_reason:
                    logger.warning(f"⚠️ Gemini blocked page {page_num}: {feedback.block_reason}")
                    return None, 0.0
            
            if text:
                # Estimate confidence
                confidence = min(len(text) / 1000.0, 1.0) if text else 0.0
                logger.info(f"✅ Gemini OCR completed for page {page_num} ({len(text)} chars)")
                return text, confidence
            else:
                logger.warning(f"⚠️ Gemini returned empty response for page {page_num}")
                logger.debug(f"   Response structure: {type(response)}")
                logger.debug(f"   Response attributes: {dir(response)}")
                return None, 0.0
        else:
            logger.warning(f"⚠️ Gemini returned no response for page {page_num}")
            return None, 0.0
            
    except Exception as e:
        logger.error(f"❌ Gemini OCR error for page {page_num}: {e}")
        logger.error(f"   Error type: {type(e).__name__}")
        logger.error(f"   Error details: {str(e)}")
        return None, 0.0

def extract_text_from_layout(layout, doc_text):
    """Extract text from a layout element."""
    if not hasattr(layout, "text_anchor") or not layout.text_anchor:
        return None
    
    segments = layout.text_anchor.text_segments
    if not segments:
        return None
    
    valid_starts = [seg.start_index for seg in segments if seg.start_index is not None]
    valid_ends = [seg.end_index for seg in segments if seg.end_index is not None]
    
    if not valid_starts or not valid_ends:
        return None
    
    return doc_text[min(valid_starts):max(valid_ends)]

def calculate_confidence_score(doc, page):
    """Calculate confidence score for Document AI OCR result."""
    # Simple heuristic: use text length and page structure
    if not doc.text:
        return 0.0
    
    # Check if page has good structure
    has_blocks = hasattr(page, "blocks") and page.blocks
    has_paragraphs = hasattr(page, "paragraphs") and page.paragraphs
    
    # Base confidence on structure quality
    confidence = 0.5  # Base confidence
    if has_blocks:
        confidence += 0.2
    if has_paragraphs:
        confidence += 0.2
    
    # Check text length (too short might indicate poor OCR)
    page_text = extract_text_from_layout(page.layout, doc.text) if hasattr(page, "layout") and page.layout else ""
    if page_text and len(page_text) > 100:
        confidence += 0.1
    
    return min(confidence, 1.0)

def parse_documentai_doc(doc, start_page=1):
    """Extract text from Document AI response with confidence scores."""
    parsed_pages = []
    for idx, page in enumerate(doc.pages):
        global_page_num = start_page + idx
        page_text = ""
        confidence = 0.0
        
        # Calculate confidence
        confidence = calculate_confidence_score(doc, page)
        
        # Extract text using best method
        if doc.text and hasattr(page, "layout") and page.layout:
            page_text = extract_text_from_layout(page.layout, doc.text)
        
        if not page_text and hasattr(page, "blocks") and page.blocks:
            block_texts = []
            for block in page.blocks:
                if hasattr(block, "layout") and block.layout:
                    block_text = extract_text_from_layout(block.layout, doc.text)
                    if block_text and block_text.strip():
                        block_texts.append(block_text)
            if block_texts:
                page_text = "\n\n".join(block_texts)
        
        if not page_text.strip():
            page_text = "[BLANK PAGE]"
        else:
            # Clean up
            lines = page_text.split('\n')
            cleaned_lines = []
            prev_empty = False
            for line in lines:
                if line.strip():
                    cleaned_lines.append(line)
                    prev_empty = False
                elif not prev_empty:
                    cleaned_lines.append('')
                    prev_empty = True
            while cleaned_lines and not cleaned_lines[-1]:
                cleaned_lines.pop()
            page_text = '\n'.join(cleaned_lines)
        
        parsed_pages.append({
            'page_num': global_page_num,
            'text': page_text,
            'confidence': confidence,
            'method': 'documentai'  # Mark as Document AI result
        })
    
    return parsed_pages

# --------------------------
# DOCUMENT AI PROCESSING
# --------------------------
def process_pdf_chunk_with_documentai(chunk_bytes, start_page=1, chunk_id=0):
    """Process PDF chunk with Document AI."""
    client_options = {"api_endpoint": f"{LOCATION}-documentai.googleapis.com"}
    client = documentai.DocumentProcessorServiceClient(client_options=client_options)
    processor_name = client.processor_path(PROJECT_ID, LOCATION, PROCESSOR_ID)

    raw_doc = documentai.RawDocument(content=chunk_bytes, mime_type="application/pdf")
    request = documentai.ProcessRequest(name=processor_name, raw_document=raw_doc)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            doc = client.process_document(request=request, timeout=REQUEST_TIMEOUT).document
            if not doc or not doc.text:
                raise ValueError("Empty document response")
            
            pages = parse_documentai_doc(doc, start_page=start_page)
            return pages, chunk_id
        except Exception as e:
            if attempt < MAX_RETRIES:
                wait = min(2 ** attempt, 10)
                logger.warning(f"⚠️ Document AI attempt {attempt} failed: {e}, retrying in {wait}s...")
                time.sleep(wait)
            else:
                logger.error(f"❌ Document AI failed after {MAX_RETRIES} retries: {e}")
                return None, chunk_id
    
    return None, chunk_id

# --------------------------
# HYBRID PROCESSING
# --------------------------
def process_page_hybrid(pdf_bytes, page_num, docai_result=None):
    """Process a single page using hybrid approach."""
    page_result = {
        'page_num': page_num,
        'text': '',
        'method': 'none',
        'confidence': 0.0
    }
    
    # If we have Document AI result, use it
    if docai_result:
        page_result['text'] = docai_result['text']
        page_result['confidence'] = docai_result['confidence']
        page_result['method'] = 'documentai'
        
        # Check if we should use Gemini for low-quality pages
        if USE_GEMINI_FOR_LOW_QUALITY and gemini_model:
            if docai_result['confidence'] < QUALITY_THRESHOLD:
                logger.info(f"📊 Page {page_num} confidence ({docai_result['confidence']:.2f}) below threshold, trying Gemini...")
                image = pdf_page_to_image(pdf_bytes, page_num - 1)  # 0-indexed
                if image:
                    gemini_text, gemini_conf = ocr_with_gemini_vision(image, page_num)
                    if gemini_text and gemini_conf > docai_result['confidence']:
                        page_result['text'] = gemini_text
                        page_result['confidence'] = gemini_conf
                        page_result['method'] = 'gemini'
                        logger.info(f"✅ Page {page_num} improved with Gemini (conf: {gemini_conf:.2f})")
    else:
        # No Document AI result, try Gemini directly
        if gemini_model:
            logger.info(f"📄 Page {page_num} not processed by Document AI, trying Gemini...")
            image = pdf_page_to_image(pdf_bytes, page_num - 1)
            if image:
                gemini_text, gemini_conf = ocr_with_gemini_vision(image, page_num)
                if gemini_text:
                    page_result['text'] = gemini_text
                    page_result['confidence'] = gemini_conf
                    page_result['method'] = 'gemini'
    
    return page_result

def split_pdf_bytes(pdf_bytes, pages_per_chunk=PAGES_PER_CHUNK):
    """Split PDF into chunks."""
    reader = PdfReader(io.BytesIO(pdf_bytes))
    total_pages = len(reader.pages)
    chunks = []

    for i in range(0, total_pages, pages_per_chunk):
        writer = PdfWriter()
        for page_num in range(i, min(i + pages_per_chunk, total_pages)):
            writer.add_page(reader.pages[page_num])
        chunk_stream = io.BytesIO()
        writer.write(chunk_stream)
        chunk_stream.seek(0)
        chunks.append((chunk_stream.read(), i + 1, i // pages_per_chunk))
    return chunks, total_pages

# --------------------------
# MAIN HYBRID PROCESS
# --------------------------
def ocr_hybrid_approach():
    """Main hybrid OCR processing function."""
    # CRITICAL: Declare globals first to access module-level variables in Jupyter
    global vectorstore, embedding_manager, GOOGLE_API_KEY, GEMINI_API_KEY, AUTHOR_NAME
    
    # Debug: Check vectorstore status immediately after global declaration
    logger.debug(f"🔍 Initial vectorstore check (after global): type={type(vectorstore)}, is not None={vectorstore is not None}")
    
    start_time = time.time()
    logger.info("=" * 60)
    logger.info("Starting Hybrid OCR: Document AI + Gemini Vision")
    logger.info("=" * 60)

    try:
        # Download PDF
        logger.info(f"Downloading PDF from {GCS_INPUT_URI}")
        storage_client = storage.Client(project=PROJECT_ID)
        bucket_name, blob_name = GCS_INPUT_URI[5:].split("/", 1)
        
        # Check if bucket exists
        bucket = storage_client.bucket(bucket_name)
        if not bucket.exists():
            raise ValueError(f"Bucket '{bucket_name}' does not exist or you don't have access to it.")
        
        # Check if blob exists
        blob = bucket.blob(blob_name)
        if not blob.exists():
            # Try with .pdf extension if missing
            if not blob_name.endswith('.pdf') and not blob_name.endswith('.PDF'):
                blob_name_with_pdf = blob_name + '.pdf'
                blob_alt = bucket.blob(blob_name_with_pdf)
                if blob_alt.exists():
                    logger.warning(f"⚠️ File '{blob_name}' not found, but found '{blob_name_with_pdf}'. Using that instead.")
                    blob = blob_alt
                    blob_name = blob_name_with_pdf
                else:
                    # List some files in the bucket to help user
                    logger.error(f"❌ File '{blob_name}' not found in bucket '{bucket_name}'.")
                    logger.error(f"   Also tried: '{blob_name_with_pdf}'")
                    logger.error(f"   Available files in bucket (first 10):")
                    blobs = list(bucket.list_blobs(max_results=10))
                    if blobs:
                        for b in blobs:
                            logger.error(f"     - {b.name}")
                    else:
                        logger.error(f"     (No files found in bucket)")
                    raise FileNotFoundError(f"File '{blob_name}' not found in bucket '{bucket_name}'. Please check the GCS_INPUT_URI.")
        
        pdf_bytes = blob.download_as_bytes()
        logger.info(f"✅ Downloaded PDF ({len(pdf_bytes)//1024} KB)")

        # Split into chunks
        chunks, total_pages = split_pdf_bytes(pdf_bytes)
        logger.info(f"📄 Split PDF into {len(chunks)} chunks ({PAGES_PER_CHUNK} pages per chunk, {total_pages} total pages)")

        # Initialize embedding manager early to ensure it's ready
        # In Jupyter notebooks, module-level variables can have scoping issues
        # So we'll check and re-initialize if needed
        vectorstore_status = "Available" if vectorstore is not None else "None"
        logger.info(f"🔍 Checking vectorstore status: {vectorstore_status}")
        
        # Re-initialize Chroma if it's None (handles Jupyter notebook scoping issues)
        # This is necessary because Jupyter can have variable scoping issues between cells
        if vectorstore is None:
            logger.warning("⚠️ Vectorstore is None in function scope, re-initializing...")
            try:
                # Get API key (check both GOOGLE_API_KEY and GEMINI_API_KEY)
                api_key = GOOGLE_API_KEY or os.getenv("GOOGLE_API_KEY") or GEMINI_API_KEY or os.getenv("GEMINI_API_KEY", "")
                if api_key:
                    logger.info(f"🔑 Using API key for re-initialization (key length: {len(api_key)} chars)")
                    gemini_embeddings = GoogleGenerativeAIEmbeddings(
                        model="models/gemini-embedding-001",
                        google_api_key=api_key
                    )
                    chroma_persist_dir = f"./chroma_db/{AUTHOR_NAME}"
                    os.makedirs(chroma_persist_dir, exist_ok=True)
                    vectorstore = Chroma(
                        persist_directory=chroma_persist_dir,
                        embedding_function=gemini_embeddings
                    )
                    logger.info(f"✅ Chroma vectorstore re-initialized successfully (persist_directory: {chroma_persist_dir})")
                else:
                    logger.error("❌ Cannot re-initialize: No API key available (check GOOGLE_API_KEY or GEMINI_API_KEY)")
            except Exception as e:
                logger.error(f"❌ Failed to re-initialize Chroma: {e}")
                import traceback
                logger.error(f"   Traceback: {traceback.format_exc()}")
        else:
            logger.info("✅ Vectorstore is available, skipping re-initialization")
        
        # Now initialize embedding manager
        # Use explicit None check instead of truthiness check
        vectorstore_type = type(vectorstore).__name__ if vectorstore is not None else "None"
        logger.info(f"🔍 Final vectorstore check - type: {vectorstore_type}, is not None: {vectorstore is not None}")
        
        if vectorstore is not None:
            logger.info("🔧 Initializing embedding manager...")
            initialize_embedding_manager()
            if embedding_manager:
                logger.info("✅ Embedding manager ready for batch processing")
            else:
                logger.error("❌ Failed to initialize embedding manager - embeddings will not work")
                logger.error("   This may indicate an issue with the BatchEmbeddingManager initialization")
        else:
            logger.error("❌ Vectorstore is None - cannot initialize embedding manager")
            logger.error("   Embeddings will be skipped. Check API key configuration.")
            logger.error("   Vectorstore was initialized at module level but is None in function scope.")

        # Step 1: Process all chunks with Document AI
        logger.info("\n🚀 Step 1: Processing with Document AI...")
        docai_results = OrderedDict()
        completed_count = 0

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {
                executor.submit(process_pdf_chunk_with_documentai, c[0], c[1], c[2]): c[2]
                for c in chunks
            }
            
            for future in as_completed(futures):
                chunk_id = futures[future]
                try:
                    pages, _ = future.result()
                    if pages:
                        for page in pages:
                            docai_results[page['page_num']] = page
                            # Store OCR results only - embeddings will be done sequentially after OCR completes
                        completed_count += 1
                        logger.info(f"✅ Chunk {chunk_id + 1}/{len(chunks)} completed ({completed_count}/{len(chunks)} total)")
                except Exception as e:
                    logger.error(f"❌ Chunk {chunk_id + 1} failed: {e}")

        # Step 2: Process low-quality pages with Gemini
        if USE_GEMINI_FOR_LOW_QUALITY and gemini_model:
            logger.info(f"\n🔍 Step 2: Checking pages with confidence < {QUALITY_THRESHOLD}...")
            low_quality_pages = [p for p in docai_results.values() if p['confidence'] < QUALITY_THRESHOLD]
            logger.info(f"   Found {len(low_quality_pages)} pages needing Gemini processing")
            
            if low_quality_pages:
                logger.info("🚀 Processing low-quality pages with Gemini Vision...")
                for page_data in low_quality_pages:
                    page_num = page_data['page_num']
                    result = process_page_hybrid(pdf_bytes, page_num, page_data)
                    docai_results[page_num] = result
                    # Store OCR results only - embeddings will be done sequentially after OCR completes
                    if result['method'] == 'gemini':
                        logger.info(f"   ✅ Page {page_num} improved with Gemini")

        # Step 3: Process any missing pages with Gemini
        missing_pages = []
        for page_num in range(1, total_pages + 1):
            if page_num not in docai_results:
                missing_pages.append(page_num)
        
        if missing_pages and gemini_model:
            logger.info(f"\n📄 Step 3: Processing {len(missing_pages)} missing pages with Gemini...")
            for page_num in missing_pages:
                result = process_page_hybrid(pdf_bytes, page_num, None)
                docai_results[page_num] = result
                # Store OCR results only - embeddings will be done sequentially after OCR completes

        # Step 4: Embed all pages sequentially (CRITICAL FIX)
        # Embeddings are done AFTER OCR is complete to avoid threading/scoping issues
        logger.info("\n🔗 Step 4: Embedding pages sequentially...")
        if vectorstore is not None and embedding_manager:
            source_file = GCS_INPUT_URI.split("/")[-1]
            total_pages_to_embed = len(docai_results)
            embedded_count = 0
            
            for page_num in sorted(docai_results.keys()):
                page = docai_results[page_num]
                if page.get('text') and page['text'].strip():
                    try:
                        embed_page_in_chroma(
                            page["text"],
                            page_num,
                            source_file=source_file
                        )
                        embedded_count += 1
                        if embedded_count % 50 == 0:
                            logger.info(f"   Progress: {embedded_count}/{total_pages_to_embed} pages embedded...")
                    except Exception as e:
                        logger.error(f"   ❌ Failed to embed page {page_num}: {e}")
            
            logger.info(f"   ✅ Finished embedding {embedded_count}/{total_pages_to_embed} pages")
            
            # Flush any remaining embeddings in the batch
            logger.info("   🔄 Flushing remaining embeddings...")
            embedding_manager.flush()
            
            # Final persist of Chroma database
            if vectorstore is not None:
                vectorstore.persist()
                logger.info("   ✅ Chroma database persisted")
        else:
            logger.warning("⚠️ Skipping embeddings - vectorstore or embedding_manager not available")
            if vectorstore is None:
                logger.warning("   ⚠️ Vectorstore is None")
            if embedding_manager is None:
                logger.warning("   ⚠️ Embedding manager is None")

        # Step 5: Final embedding statistics
        logger.info("\n📊 Step 5: Embedding Statistics...")
        if embedding_manager:
            embed_stats = embedding_manager.get_stats()
            logger.info(f"   ✅ Embedded: {embed_stats['embedded']} pages")
            if embed_stats['failed'] > 0:
                logger.warning(f"   ⚠️ Failed: {embed_stats['failed']} pages")
            if embed_stats['pending'] > 0:
                logger.warning(f"   ⚠️ Pending: {embed_stats['pending']} pages")
        else:
            logger.warning("⚠️ Embedding manager not initialized, no embeddings were created")
            if vectorstore is None:
                logger.error("❌ Chroma vectorstore was not initialized. Check GOOGLE_API_KEY and embedding initialization.")
            else:
                logger.error("❌ Embedding manager failed to initialize even though vectorstore is available")
                logger.error("   This may indicate an issue with the embedding initialization")

        # Step 6: Combine results
        logger.info("\n📝 Step 6: Combining results...")
        combined_text = ""
        docai_count = 0
        gemini_count = 0
        
        for page_num in sorted(docai_results.keys()):
            page_data = docai_results[page_num]
            method = page_data.get('method', 'unknown')  # Use .get() with default for safety
            if method == 'documentai':
                docai_count += 1
            elif method == 'gemini':
                gemini_count += 1
            
            combined_text += f"\n--- PAGE {page_num} ---\n{page_data['text']}\n--- END PAGE {page_num} ---\n"

        # Save OCR text results to file
        store_document(OUTPUT_TXT_PATH, combined_text)
        logger.info(f"✅ OCR text saved to: {OUTPUT_TXT_PATH}")
        
        elapsed = time.time() - start_time
        logger.info("=" * 60)
        logger.info(f"✅ Hybrid OCR processing complete!")
        logger.info(f"   OCR text output: {OUTPUT_TXT_PATH}")
        logger.info(f"   Total time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")
        logger.info(f"   Pages processed: {len(docai_results)}/{total_pages}")
        logger.info(f"   Document AI: {docai_count} pages")
        logger.info(f"   Gemini Vision: {gemini_count} pages")
        logger.info(f"   Average confidence: {sum(p['confidence'] for p in docai_results.values())/len(docai_results):.2f}")
        # Final embedding summary
        logger.info("\n📊 Embedding Summary:")
        # Use explicit None check to avoid truthiness evaluation issues
        if vectorstore is not None:
            logger.info("   ✅ Chroma vectorstore: Available")
            if embedding_manager:
                embed_stats = embedding_manager.get_stats()
                logger.info(f"   ✅ Pages embedded in Chroma: {embed_stats['embedded']}")
                logger.info(f"   📁 Chroma database location: ./chroma_db/{AUTHOR_NAME}")
                if embed_stats['failed'] > 0:
                    logger.warning(f"   ⚠️ Failed embeddings: {embed_stats['failed']}")
                if embed_stats['pending'] > 0:
                    logger.warning(f"   ⚠️ Pending embeddings: {embed_stats['pending']}")
                
                # Verify embeddings by checking Chroma collection
                try:
                    collection = vectorstore._collection
                    count = collection.count()
                    logger.info(f"   📊 Total documents in Chroma: {count}")
                    if count == 0:
                        logger.error("   ❌ WARNING: Chroma database is empty! Embeddings may have failed.")
                    else:
                        logger.info(f"   ✅ Successfully embedded {count} documents in Chroma")
                except Exception as e:
                    logger.warning(f"   ⚠️ Could not verify Chroma collection: {e}")
            else:
                logger.error("   ❌ Embedding manager was not initialized")
                logger.error("   ❌ No pages were embedded in Chroma")
        else:
            logger.error("   ❌ Chroma vectorstore was not initialized")
            logger.error("   ❌ Check GOOGLE_API_KEY or GEMINI_API_KEY configuration")
            logger.error(f"   Debug: vectorstore type = {type(vectorstore)}, value = {vectorstore}")
        logger.info("=" * 60)

    except Exception as e:
        logger.error(f"❌ Fatal error: {e}", exc_info=True)
        raise

# --------------------------
# RUN
# --------------------------
if __name__ == "__main__":
    # Install required packages if needed:
    # pip install google-generativeai pdf2image pillow
    
    ocr_hybrid_approach()
